# exp135_tvt_dense_high_drift_confidence_gate_on_exp092 train

exp092 lgb1 を base に、tvt_dense 系候補を high-drift / high-disagreement regime だけで使う posthoc gate を評価する。LightGBM の新規学習は行わない。

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json

import pandas as pd

from settings import ExperimentPaths
from tvt_dense_high_drift_confidence_gate_on_exp092 import OUTPUT_PREFIX

In [ ]:
paths = ExperimentPaths()
config = paths.config
output_dir = paths.artifacts_dir
output_dir.mkdir(parents=True, exist_ok=True)

print('experiment:', config['experiment']['name'])
print('route:', config['experiment']['route'])
print('parent:', config['lineage']['parent'])
print('audit mode:', config['audit']['mode'])
print('output_dir:', output_dir)
print('gate variants:', len(config['audit']['gate_variants']))

## 2. Input contract

In [ ]:
input_contract = {
    'exp092_predictions': config['data']['exp092_predictions'],
    'exp073_predictions': config['data']['exp073_predictions'],
    'exp072_feature_cache': config['data']['exp072_feature_cache'],
    'required_rawtest_compatible_columns': config['audit']['required_rawtest_compatible_columns'],
}
print(json.dumps(input_contract, indent=2))

## 3. Run posthoc gate audit

In [ ]:
from tvt_dense_high_drift_confidence_gate_on_exp092 import run_train_from_config

summary = run_train_from_config(config, output_dir=output_dir)
print(json.dumps(summary['base'], indent=2))
print(json.dumps(summary['best'], indent=2))
print(json.dumps(summary['oracle'], indent=2)[:2000])

## 4. Metrics and artifacts

In [ ]:
metrics_path = output_dir / f'{OUTPUT_PREFIX}_metrics.csv'
gate_path = output_dir / f'{OUTPUT_PREFIX}_gate_variants.csv'
common_path = output_dir / f'{OUTPUT_PREFIX}_common_worst_metrics.csv'
bucket_path = output_dir / f'{OUTPUT_PREFIX}_bucket_metrics.csv'
parity_path = output_dir / f'{OUTPUT_PREFIX}_rawtest_parity_checklist.csv'

metrics = pd.read_csv(metrics_path).sort_values('rmse')
gate = pd.read_csv(gate_path)
common = pd.read_csv(common_path)
bucket = pd.read_csv(bucket_path)
parity = pd.read_csv(parity_path)

display(metrics.head(15))
display(gate)
display(common.sort_values(['set_name', 'delta_rmse_vs_exp092']).head(30))
display(bucket.sort_values('base_exp092_lgb1_rmse', ascending=False).head(20))
display(parity)